# Lip2Wav (PyTorch) — Reproducing SOTA Results

> **CVPR 2020** · PyTorch port by [joannahong](https://github.com/joannahong/Lip2Wav-pytorch)

**Before running:** Go to **Runtime → Change runtime type → T4 GPU** → Save.

---
### The notebook tries to run the inference of the Lip2Wav model (https://github.com/Rudrabha/Lip2Wav) on a video file and save the output as a wav file.

### But, unfortunately, it is failing at preprocessing. It is not able to detect any clips with faces.

### Need to fix this.

## Cell 1 — Check GPU

In [17]:
import shutil, sys, torch

# Check for GPU using PyTorch (more reliable than nvidia-smi)
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    nvsmi = shutil.which('nvidia-smi')
    if nvsmi:
        import subprocess
        print(subprocess.run([nvsmi], capture_output=True, text=True).stdout)
    else:
        print('No GPU detected! Go to Runtime > Change runtime type > T4 GPU')
        sys.exit()

print(f'Python {sys.version.split()[0]} | PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

GPU: Tesla T4
VRAM: 14.6 GB
Python 3.12.12 | PyTorch 2.10.0+cu128 | CUDA: True


## Cell 2 — System & Python Dependencies

In [18]:
%%bash
apt-get update -qq && apt-get install -y ffmpeg libsndfile1 > /dev/null 2>&1
echo "ffmpeg: $(ffmpeg -version 2>&1 | head -1)"
pip install -q librosa soundfile scipy pillow inflect Unidecode tqdm opencv-python yt-dlp
pip install -q pesq pystoi  # for evaluation metrics
echo "Python packages installed"

ffmpeg: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
Python packages installed


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## Cell 3 — Clone Both Repos

We need both:
- **Original Lip2Wav** (Rudrabha): contains `Dataset/chem/test.txt` with YouTube IDs, `preprocess.py`, `face_detection/`
- **Lip2Wav-pytorch** (joannahong): contains the PyTorch model, `test.py` for inference

In [19]:
import os, sys

PYTORCH_REPO = '/content/Lip2Wav-pytorch'
ORIG_REPO    = '/content/Lip2Wav-orig'

if not os.path.isdir(PYTORCH_REPO):
    !git clone https://github.com/joannahong/Lip2Wav-pytorch.git {PYTORCH_REPO}
else:
    print('PyTorch repo already cloned')

if not os.path.isdir(ORIG_REPO):
    !git clone https://github.com/Rudrabha/Lip2Wav.git {ORIG_REPO}
else:
    print('Original repo already cloned')

# Copy face_detection into PyTorch repo
if not os.path.isdir(os.path.join(PYTORCH_REPO, 'face_detection')):
    !cp -r {ORIG_REPO}/face_detection {PYTORCH_REPO}/face_detection
    print('face_detection module copied')

os.chdir(PYTORCH_REPO)
sys.path.insert(0, PYTORCH_REPO)
print('Working directory:', os.getcwd())

# Verify test.txt exists
test_txt_path = os.path.join(ORIG_REPO, 'Dataset', 'chem', 'test.txt')
print(f'\ntest.txt exists: {os.path.exists(test_txt_path)}')
if os.path.exists(test_txt_path):
    with open(test_txt_path) as f:
        content = f.read().strip()
    print(f'Contents:\n{content}')
else:
    # Search for it
    print('Searching for test.txt...')
    !find {ORIG_REPO} -name '*.txt' -path '*/chem/*' 2>/dev/null
    !find {ORIG_REPO} -name '*.txt' -path '*/Dataset/*' 2>/dev/null
    !ls -laR {ORIG_REPO}/Dataset/ 2>/dev/null | head -40

PyTorch repo already cloned
Original repo already cloned
Working directory: /content/Lip2Wav-pytorch

test.txt exists: True
Contents:
wrgnaV-cVrs
QL6TFnaXvMg
vC2v1n-bZQU
AGZZGR5Otxw
4kxMdkzCJGA
VzvinAckmQU
lItaSuF6Uqg
V_0Bnc_URos
xcJkUZ4fzEE
oYHyZarcCQQ
U6uWvuiO1rA
JIKXNx5TK3Q
FeJ1LQe6Zh4
1yoOSIcsCyI
Z08pNaGQPv8
hSBT6oe6dSA


## Cell 4 — Download s3fd Face Detection Model

In [20]:
import os

sfd_path = os.path.join(PYTORCH_REPO, 'face_detection', 'detection', 'sfd', 's3fd.pth')
os.makedirs(os.path.dirname(sfd_path), exist_ok=True)
if not os.path.exists(sfd_path):
    !wget -q --show-progress -O {sfd_path} \
        'https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth'
    print(f's3fd model: {os.path.getsize(sfd_path)//1024//1024} MB')
else:
    print('s3fd already present')

s3fd already present


## Cell 4b — Patch audio_v.py (librosa 0.10+ & NumPy 2.0 compat)

In [21]:
import re, os

def patch_audio_file(fname):
    if not os.path.exists(fname): return
    with open(fname) as f: src = f.read()
    p = src
    p = re.sub(r'librosa\.filters\.mel\(\s*(hparams\.[a-z_]+)\s*,\s*(hparams\.[a-z_]+)',
               r'librosa.filters.mel(sr=\1, n_fft=\2', p)
    p = p.replace('.astype(np.complex)', '.astype(np.complex128)')
    p = p.replace('(np.complex)', '(np.complex128)')
    p = re.sub(r'\bnp\.float\b', 'np.float64', p)
    if p != src:
        with open(fname, 'w') as f: f.write(p)
        print(f'Patched: {fname}')
    else:
        print(f'Already OK: {fname}')

for fname in [os.path.join(PYTORCH_REPO, 'utils', 'audio_v.py'),
              os.path.join(PYTORCH_REPO, 'utils', 'audio.py')]:
    patch_audio_file(fname)

import librosa, numpy as np
print(f'librosa {librosa.__version__} | numpy {np.__version__}')

Already OK: /content/Lip2Wav-pytorch/utils/audio_v.py
Already OK: /content/Lip2Wav-pytorch/utils/audio.py
librosa 0.11.0 | numpy 2.0.2


## Cell 5 — Download Chem Checkpoint

In [22]:
import os, glob, torch

CKPT_DIR = os.path.join(PYTORCH_REPO, 'checkpoints')
ZIP_PATH = os.path.join(CKPT_DIR, 'chem_checkpoint.zip')
os.makedirs(CKPT_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print('Downloading checkpoint (~456 MB)...')
    !wget -q --show-progress \
        'https://www.dropbox.com/sh/p6ljz9knhegxudl/AAAe63m0mpOZjUDbXbmkKROla?dl=1' \
        -O {ZIP_PATH}
else:
    print('Checkpoint already downloaded')

!unzip -q -o {ZIP_PATH} -d {CKPT_DIR}

candidates = [
    p for p in glob.glob(os.path.join(CKPT_DIR, '**', '*'), recursive=True)
    if os.path.isfile(p) and not p.endswith('.zip') and os.path.getsize(p) > 1_000_000
]
ckpt_files = [c for c in candidates if 'ckpt_' in os.path.basename(c)]
CKPT_PATH = ckpt_files[0] if ckpt_files else candidates[0]
print(f'Checkpoint: {CKPT_PATH}')

ckpt = torch.load(CKPT_PATH, map_location='cpu')
print(f'Keys: {list(ckpt.keys())} | Iteration: {ckpt.get("iteration", "?")}')

Checkpoint already downloaded
mapname:  conversion of  failed
Checkpoint: /content/Lip2Wav-pytorch/checkpoints/chem/ckpt_223000
Keys: ['model', 'optimizer', 'iteration'] | Iteration: 223000


## Cell 6 — Download Chem Test Videos & Create Intervals

Read the YouTube IDs from `Dataset/chem/test.txt` in the original repo,
download each video, and split into 30-second intervals to match the
dataset structure expected by `preprocess.py`.

**The model is speaker-specific** — it only works for this exact speaker.

In [23]:
import os, subprocess

# ── Dataset directory (will hold intervals/ and preprocessed/) ──
DATASET_DIR = '/content/Dataset/chem'
INTERVALS_DIR = os.path.join(DATASET_DIR, 'intervals')
VIDEO_DIR = os.path.join(DATASET_DIR, 'videos')
os.makedirs(INTERVALS_DIR, exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)

# ── Read test.txt ──
test_txt = os.path.join(ORIG_REPO, 'Dataset', 'chem', 'test.txt')
test_ids = []

if os.path.exists(test_txt):
    with open(test_txt) as f:
        test_ids = [line.strip() for line in f if line.strip()]
    print(f'Found {len(test_ids)} test video IDs: {test_ids}')
else:
    # If test.txt is not found, search the entire cloned repo
    import glob as g
    txt_files = g.glob(os.path.join(ORIG_REPO, '**', '*.txt'), recursive=True)
    print(f'test.txt not at expected path. Found .txt files:')
    for tf in txt_files:
        print(f'  {tf}')
        with open(tf) as f:
            print(f'    -> {f.read().strip()[:200]}')

if not test_ids:
    raise RuntimeError(
        'No test video IDs found! Check that the original Lip2Wav repo '
        'was cloned successfully and contains Dataset/chem/test.txt. '
        'You can also manually set test_ids = ["VIDEO_ID_1", "VIDEO_ID_2"] '
        'with the correct YouTube IDs for the chem speaker.'
    )

Found 16 test video IDs: ['wrgnaV-cVrs', 'QL6TFnaXvMg', 'vC2v1n-bZQU', 'AGZZGR5Otxw', '4kxMdkzCJGA', 'VzvinAckmQU', 'lItaSuF6Uqg', 'V_0Bnc_URos', 'xcJkUZ4fzEE', 'oYHyZarcCQQ', 'U6uWvuiO1rA', 'JIKXNx5TK3Q', 'FeJ1LQe6Zh4', '1yoOSIcsCyI', 'Z08pNaGQPv8', 'hSBT6oe6dSA']


In [24]:
import os, subprocess

# ── Download test videos and create 30-second intervals ──
# We process up to 3 videos for a manageable reproduction run
N_TEST_VIDEOS = 3
INTERVAL_SEC = 30
downloaded_intervals = []

for vid_id in test_ids[:N_TEST_VIDEOS]:
    vid_path = os.path.join(VIDEO_DIR, f'{vid_id}.mp4')
    
    # Download
    if not os.path.exists(vid_path):
        print(f'\nDownloading {vid_id}...')
        result = subprocess.run([
            'yt-dlp',
            '-f', 'bestvideo[ext=mp4][height<=480]+bestaudio[ext=m4a]/best[ext=mp4]',
            '--merge-output-format', 'mp4',
            '-o', vid_path,
            f'https://www.youtube.com/watch?v={vid_id}'
        ], capture_output=True, text=True)
        if result.returncode != 0:
            print(f'  FAILED: {result.stderr[:300]}')
            continue
    
    if not os.path.exists(vid_path):
        print(f'  Skipping {vid_id} - not found after download')
        continue
    
    # Get duration
    dur = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'default=noprint_wrappers=1:nokey=1', vid_path],
        capture_output=True, text=True
    )
    try:
        duration = float(dur.stdout.strip())
    except:
        print(f'  Could not get duration for {vid_id}')
        continue
    
    print(f'  {vid_id}: {duration:.1f}s')
    
    # Create 30s interval clips in: intervals/{vid_id}/{vid_id}_{start}.mp4
    # This matches the original Lip2Wav structure where intervals are
    # organized under the video ID directory
    vid_interval_dir = os.path.join(INTERVALS_DIR, vid_id)
    os.makedirs(vid_interval_dir, exist_ok=True)
    
    for start in range(0, int(duration) - INTERVAL_SEC + 1, INTERVAL_SEC):
        interval_name = f'{vid_id}_{start:06d}'
        interval_subdir = os.path.join(vid_interval_dir, interval_name)
        interval_vid = os.path.join(interval_subdir, f'{interval_name}.mp4')
        
        if os.path.exists(interval_vid):
            downloaded_intervals.append(interval_subdir)
            continue
        
        os.makedirs(interval_subdir, exist_ok=True)
        subprocess.run([
            'ffmpeg', '-y', '-loglevel', 'error',
            '-ss', str(start), '-i', vid_path,
            '-t', str(INTERVAL_SEC),
            '-c', 'copy',
            interval_vid
        ], capture_output=True)
        
        if os.path.exists(interval_vid) and os.path.getsize(interval_vid) > 10000:
            downloaded_intervals.append(interval_subdir)

print(f'\nTotal intervals created: {len(downloaded_intervals)}')
if len(downloaded_intervals) == 0:
    print('WARNING: No intervals created. Check YouTube IDs and network.')
else:
    print(f'First 5: {[os.path.basename(d) for d in downloaded_intervals[:5]]}')

  wrgnaV-cVrs: 137.9s
  QL6TFnaXvMg: 183.6s
  vC2v1n-bZQU: 290.2s

Total intervals created: 19
First 5: ['wrgnaV-cVrs_000000', 'wrgnaV-cVrs_000030', 'wrgnaV-cVrs_000060', 'wrgnaV-cVrs_000090', 'QL6TFnaXvMg_000000']


## Cell 6b — Preprocess: Face Detection + Crop + Audio Extraction

Creates the directory layout that `test.py`'s `Generator` class expects:
```
Dataset/chem/preprocessed/{vid_id}/{interval_name}/
    0.jpg  1.jpg  2.jpg  ...  (96x96 face crops, consecutively numbered)
    audio/audio.wav             (16kHz mono)
```

**Key fixes vs previous version:**
1. Frame filenames must be **consecutive integers starting from 0** (not zero-padded) — 
   `contiguous_window_generator` in test.py parses `int(basename)` and checks for gaps
2. When a face is missed, we **interpolate from the nearest detected face** instead of
   skipping — this preserves consecutive numbering and keeps frame count >= T=90
3. We use a **confidence threshold** and process frames individually when batch detection
   fails on small faces

In [25]:
import os, sys, cv2, subprocess, shutil
import numpy as np
import torch
from glob import glob
from tqdm import tqdm

os.chdir(PYTORCH_REPO)
sys.path.insert(0, PYTORCH_REPO)

from hparams import hparams as hps
import face_detection

print(f'hparams: fps={hps.fps}, T={hps.T}, overlap={hps.overlap}, '
      f'img_size={hps.img_size}, sample_rate={hps.sample_rate}')

detector = face_detection.FaceAlignment(
    face_detection.LandmarksType._2D, device='cuda', flip_input=False
)

PREPROCESSED_DIR = os.path.join(DATASET_DIR, 'preprocessed')
# Clear previous failed preprocessing
shutil.rmtree(PREPROCESSED_DIR, ignore_errors=True)
os.makedirs(PREPROCESSED_DIR, exist_ok=True)

valid_intervals = []
skipped_reasons = {}

for interval_subdir in tqdm(downloaded_intervals, desc='Preprocessing'):
    interval_name = os.path.basename(interval_subdir)
    vid_id = os.path.basename(os.path.dirname(interval_subdir))

    out_dir = os.path.join(PREPROCESSED_DIR, vid_id, interval_name)
    audio_dir = os.path.join(out_dir, 'audio')

    # Skip if already preprocessed
    existing = glob(os.path.join(out_dir, '*.jpg'))
    if os.path.isdir(out_dir) and len(existing) >= hps.T:
        valid_intervals.append(out_dir)
        continue

    vid_files = glob(os.path.join(interval_subdir, '*.mp4'))
    if not vid_files:
        skipped_reasons[interval_name] = 'no mp4 found'
        continue
    vid_path = vid_files[0]

    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(audio_dir, exist_ok=True)

    # ---- Extract audio ----
    audio_path = os.path.join(audio_dir, 'audio.wav')
    subprocess.run([
        'ffmpeg', '-y', '-loglevel', 'error',
        '-i', vid_path, '-vn', '-acodec', 'pcm_s16le',
        '-ar', str(hps.sample_rate), '-ac', '1', audio_path
    ], capture_output=True)

    if not os.path.exists(audio_path) or os.path.getsize(audio_path) < 1000:
        shutil.rmtree(out_dir, ignore_errors=True)
        skipped_reasons[interval_name] = 'audio extraction failed'
        continue

    # ---- Extract frames ----
    tmp_frames = '/tmp/lip2wav_frames'
    shutil.rmtree(tmp_frames, ignore_errors=True)
    os.makedirs(tmp_frames)

    subprocess.run([
        'ffmpeg', '-y', '-loglevel', 'error',
        '-i', vid_path, '-vf', f'fps={hps.fps}',
        '-q:v', '2', os.path.join(tmp_frames, '%05d.jpg')
    ], capture_output=True)

    frame_paths = sorted(glob(os.path.join(tmp_frames, '*.jpg')))
    n_frames = len(frame_paths)
    if n_frames < hps.T:
        shutil.rmtree(out_dir, ignore_errors=True)
        shutil.rmtree(tmp_frames, ignore_errors=True)
        skipped_reasons[interval_name] = f'only {n_frames} frames (need {hps.T})'
        continue

    # ---- Face detection ----
    # Process in batches. The s3fd detector's get_detections_for_batch()
    # returns a list where each element is either:
    #   - a numpy array of shape (N, 5) with [x1, y1, x2, y2, confidence]
    #   - None if no face detected
    #   - an empty array
    # We also try single-frame detection as fallback for missed frames.
    BATCH_SZ = 16
    face_crops = {}  # idx -> 96x96 BGR image
    batch_imgs, batch_indices = [], []

    for idx, fp in enumerate(frame_paths):
        img = cv2.imread(fp)
        if img is None:
            continue
        batch_imgs.append(img)
        batch_indices.append(idx)

        if len(batch_imgs) == BATCH_SZ or idx == len(frame_paths) - 1:
            try:
                preds = detector.get_detections_for_batch(np.array(batch_imgs))
            except Exception as e:
                preds = [None] * len(batch_imgs)

            for b_idx, (pred, orig_idx) in enumerate(zip(preds, batch_indices)):
                try:
                    # Handle various return formats from s3fd
                    if pred is None:
                        continue
                    coords = np.array(pred, dtype=float)
                    if coords.ndim == 0 or coords.size == 0:
                        continue
                    # If multiple detections, take the one with highest confidence
                    if coords.ndim == 2 and coords.shape[0] > 1:
                        best = np.argmax(coords[:, 4]) if coords.shape[1] > 4 else 0
                        coords = coords[best]
                    coords = coords.flatten()
                    if len(coords) < 4:
                        continue

                    x1 = int(max(0, coords[0]))
                    y1 = int(max(0, coords[1]))
                    x2 = int(min(batch_imgs[b_idx].shape[1], coords[2]))
                    y2 = int(min(batch_imgs[b_idx].shape[0], coords[3]))

                    if x2 <= x1 or y2 <= y1:
                        continue

                    crop = batch_imgs[b_idx][y1:y2, x1:x2]
                    if crop.size > 0 and crop.shape[0] > 5 and crop.shape[1] > 5:
                        crop = cv2.resize(crop, (hps.img_size, hps.img_size))
                        face_crops[orig_idx] = crop
                except:
                    pass

            batch_imgs, batch_indices = [], []

    # ---- Fallback: try single-frame detection for missed frames ----
    # MIT lecture videos have small faces that batch detection sometimes misses
    missed_count = n_frames - len(face_crops)
    if 0 < missed_count < n_frames and missed_count > n_frames * 0.3:
        tqdm.write(f'  {interval_name}: batch detected {len(face_crops)}/{n_frames}, '
                   f'trying single-frame fallback for {missed_count} missed...')
        for idx in range(n_frames):
            if idx in face_crops:
                continue
            fp = frame_paths[idx]
            img = cv2.imread(fp)
            if img is None:
                continue
            try:
                preds = detector.get_detections_for_batch(np.array([img]))
                pred = preds[0] if preds else None
                if pred is not None:
                    coords = np.array(pred, dtype=float)
                    if coords.ndim == 0 or coords.size == 0:
                        continue
                    if coords.ndim == 2:
                        best = np.argmax(coords[:, 4]) if coords.shape[1] > 4 else 0
                        coords = coords[best]
                    coords = coords.flatten()
                    if len(coords) < 4:
                        continue
                    x1, y1 = int(max(0, coords[0])), int(max(0, coords[1]))
                    x2, y2 = int(min(img.shape[1], coords[2])), int(min(img.shape[0], coords[3]))
                    if x2 > x1 and y2 > y1:
                        crop = img[y1:y2, x1:x2]
                        if crop.size > 0 and crop.shape[0] > 5 and crop.shape[1] > 5:
                            face_crops[idx] = cv2.resize(crop, (hps.img_size, hps.img_size))
            except:
                pass

    detect_rate = len(face_crops) / max(n_frames, 1) * 100

    # ---- Fill remaining missing frames by nearest-neighbor interpolation ----
    # test.py's contiguous_window_generator needs consecutive frame numbers
    if len(face_crops) > 0:
        detected_indices = sorted(face_crops.keys())
        for idx in range(n_frames):
            if idx not in face_crops:
                nearest = min(detected_indices, key=lambda d: abs(d - idx))
                face_crops[idx] = face_crops[nearest]

    # ---- Save as consecutively numbered JPGs (0.jpg, 1.jpg, ...) ----
    # test.py parses filenames as int(os.path.splitext(basename)[0])
    if len(face_crops) >= hps.T:
        saved = 0
        for idx in range(n_frames):
            if idx in face_crops:
                cv2.imwrite(os.path.join(out_dir, f'{idx}.jpg'), face_crops[idx])
                saved += 1
        valid_intervals.append(out_dir)
        tqdm.write(f'  {interval_name}: {saved} frames saved, '
                   f'detect_rate={detect_rate:.0f}%')
    else:
        shutil.rmtree(out_dir, ignore_errors=True)
        skipped_reasons[interval_name] = (
            f'only {len(face_crops)} face crops (need {hps.T}), '
            f'detect_rate={detect_rate:.0f}%')

    shutil.rmtree(tmp_frames, ignore_errors=True)

del detector
torch.cuda.empty_cache()

print(f'\n{"="*60}')
print(f'Valid preprocessed intervals: {len(valid_intervals)} / {len(downloaded_intervals)}')
for v in valid_intervals[:5]:
    n_imgs = len(glob(os.path.join(v, '*.jpg')))
    has_audio = os.path.exists(os.path.join(v, 'audio', 'audio.wav'))
    print(f'  {os.path.relpath(v, PREPROCESSED_DIR)}: {n_imgs} frames, audio={has_audio}')

if skipped_reasons:
    print(f'\nSkipped intervals ({len(skipped_reasons)}):')
    for name, reason in list(skipped_reasons.items())[:10]:
        print(f'  {name}: {reason}')

print(f'\nPreprocessed tree:')
!find {PREPROCESSED_DIR} -maxdepth 3 -type d | head -20

hparams: fps=30, T=90, overlap=15, img_size=96, sample_rate=16000


Preprocessing: 100%|██████████| 19/19 [15:23<00:00, 48.59s/it]


Valid preprocessed intervals: 0 / 19

Skipped intervals (19):
  wrgnaV-cVrs_000000: only 0 face crops (need 90), detect_rate=0%
  wrgnaV-cVrs_000030: only 0 face crops (need 90), detect_rate=0%
  wrgnaV-cVrs_000060: only 0 face crops (need 90), detect_rate=0%
  wrgnaV-cVrs_000090: only 0 face crops (need 90), detect_rate=0%
  QL6TFnaXvMg_000000: only 0 face crops (need 90), detect_rate=0%
  QL6TFnaXvMg_000030: only 0 face crops (need 90), detect_rate=0%
  QL6TFnaXvMg_000060: only 0 face crops (need 90), detect_rate=0%
  QL6TFnaXvMg_000090: only 0 face crops (need 90), detect_rate=0%
  QL6TFnaXvMg_000120: only 0 face crops (need 90), detect_rate=0%
  QL6TFnaXvMg_000150: only 0 face crops (need 90), detect_rate=0%

Preprocessed tree:
/content/Dataset/chem/preprocessed
/content/Dataset/chem/preprocessed/wrgnaV-cVrs
/content/Dataset/chem/preprocessed/QL6TFnaXvMg
/content/Dataset/chem/preprocessed/vC2v1n-bZQU


## Cell 7 — Run Inference via test.py

The PyTorch port's `test.py` accepts `--data_dir` pointing to the dataset root.
It internally does:
```python
glob(os.path.join(data_dir, 'preprocessed', vid_id, '*', '*.jpg'))
```
We call it directly to ensure exact reproduction.

In [26]:
import os, sys

os.chdir(PYTORCH_REPO)
RESULTS_DIR = '/content/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# First, let's inspect test.py to see exactly what arguments it expects
# and how it loads data, so we can verify our directory structure matches.
print('=== test.py contents (key sections) ===')
with open(os.path.join(PYTORCH_REPO, 'test.py')) as f:
    source = f.read()
print(source[:5000])
print('\n... (truncated) ...')

=== test.py contents (key sections) ===
# from inference import load_model, infer_vid
import torch
from shutil import copy
from torchvision import transforms
from PIL import Image
from pathlib import Path
from glob import glob
import numpy as np
from tqdm import tqdm
import sys, cv2, os, pickle, argparse, subprocess

from model.model import Tacotron2
from hparams import hparams as hps
import utils.audio_v as audio
from utils.util import mode, to_var, to_arr

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

torch.manual_seed(0)
torch.cuda.manual_seed_all(0)

class Generator(object):
	def __init__(self, model):
		super(Generator, self).__init__()

		self.synthesizer = model.eval()


	def read_window(self, window_fnames):
		window = []
		for fname in window_fnames:
			img = cv2.imread(fname)
			height, width, channels = img.shape
			path = Path(fname)
			fparent = str(path.parent.parent.parent)
			############ transform
			if fparent[-3:] == 'new':
				img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


In [27]:
import os, sys, re

os.chdir(PYTORCH_REPO)
sys.path.insert(0, PYTORCH_REPO)

# Read test.py source to understand glob pattern and adapt if needed
with open(os.path.join(PYTORCH_REPO, 'test.py')) as f:
    test_source = f.read()

# Find the glob pattern used to locate preprocessed data
glob_matches = re.findall(r'glob\.glob\(.*?\)', test_source, re.DOTALL)
print('Glob patterns found in test.py:')
for m in glob_matches:
    print(f'  {m}')

# Find the argparse arguments
arg_matches = re.findall(r'add_argument\(.*?\)', test_source, re.DOTALL)
print('\nArgparse arguments:')
for m in arg_matches:
    print(f'  {m}')

Glob patterns found in test.py:

Argparse arguments:
  add_argument('-d', "--data_dir", help="Speaker folder path", required=True)
  add_argument('-r', "--results_dir", help="Speaker folder path", required=True)
  add_argument('--checkpoint', help="Path to trained checkpoint", required=True)
  add_argument("--preset", help="Speaker-specific hyper-params", type=str, required=False)


## Cell 7b — Run Inference (adapted to actual test.py structure)

Based on what we learned from inspecting test.py above, run inference.
If test.py can be called directly via CLI, we do that. Otherwise we
replicate its logic exactly.

In [28]:
import sys, os, cv2, shutil
import numpy as np
import torch
from glob import glob
from tqdm import tqdm

os.chdir(PYTORCH_REPO)
sys.path.insert(0, PYTORCH_REPO)

RESULTS_DIR = '/content/results'
shutil.rmtree(RESULTS_DIR, ignore_errors=True)
os.makedirs(os.path.join(RESULTS_DIR, 'wavs'), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'gts'), exist_ok=True)

# ── Reload modules cleanly ──
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['audio', 'model', 'test', 'hparam']):
        del sys.modules[mod]

from hparams import hparams as hps
from model.model import Tacotron2
from utils.util import mode
import utils.audio_v as audio
from test import infer_vid

print(f'hparams: fps={hps.fps}, T={hps.T}, overlap={hps.overlap}, '
      f'img_size={hps.img_size}, mel_overlap={hps.mel_overlap}')

# ── Load model ──
print('Loading model...')
ckpt_dict = torch.load(CKPT_PATH)
model = Tacotron2()
model.load_state_dict(ckpt_dict['model'])
model = mode(model, True).eval()
print(f'Model loaded (iteration {ckpt_dict["iteration"]})')

# ── read_window: exact replica of test.py Generator.read_window() ──
# Note: test.py checks path.parent.parent.parent ending with 'new'
# Our paths are under 'preprocessed' (not 'new'), so the transform
# branch is NOT taken — same as the original chem data.
def read_window(window_fnames):
    window = []
    for fname in window_fnames:
        img = cv2.imread(fname)
        if img is None:
            return None
        img = cv2.resize(img, (hps.img_size, hps.img_size))
        window.append(img)
    return np.asarray(window) / 255.

# ── Griffin-Lim fix for modern numpy ──
def fixed_griffin_lim(S, hparams):
    angles = np.exp(2j * np.pi * np.random.rand(*S.shape))
    S_complex = np.abs(S).astype(np.complex128)
    y = audio._istft(S_complex * angles, hparams)
    for _ in range(hparams.griffin_lim_iters):
        angles = np.exp(1j * np.angle(audio._stft(y, hparams)))
        y = audio._istft(S_complex * angles, hparams)
    return y
audio._griffin_lim = fixed_griffin_lim

# ── Synthesize each interval ──
# We replicate test.py's contiguous_window_generator + vc() logic:
# 1. Parse frame filenames as integers
# 2. Find contiguous runs of consecutive IDs
# 3. For each run >= T frames, do sliding-window inference
print(f'\nProcessing {len(valid_intervals)} intervals...\n')

for interval_dir in tqdm(valid_intervals, desc='Inference'):
    interval_name = os.path.basename(interval_dir)
    wav_out = os.path.join(RESULTS_DIR, 'wavs', f'{interval_name}.wav')
    gt_out  = os.path.join(RESULTS_DIR, 'gts',  f'{interval_name}.wav')

    if os.path.exists(wav_out):
        continue

    # Get all JPGs and sort by their integer filename
    jpg_files = glob(os.path.join(interval_dir, '*.jpg'))
    if len(jpg_files) < hps.T:
        print(f'  Skipped {interval_name} ({len(jpg_files)} < {hps.T} frames)')
        continue

    # Sort by integer index (matching test.py's parsing)
    jpg_files = sorted(jpg_files,
                       key=lambda f: int(os.path.splitext(os.path.basename(f))[0]))

    # Build the full image list for vc()
    all_windows = []
    i = 0
    while i + hps.T <= len(jpg_files):
        all_windows.append(jpg_files[i : i + hps.T])
        i += hps.T - hps.overlap
    remainder = jpg_files[i:]
    if len(remainder) > 0:
        all_windows.append(remainder)

    mel = None
    with torch.no_grad():
        for w_idx, w_fnames in enumerate(all_windows):
            if len(w_fnames) == 0:
                continue
            # Pad short final window by repeating last frame
            while len(w_fnames) < hps.T:
                w_fnames = w_fnames + [w_fnames[-1]]

            images = read_window(w_fnames)
            if images is None:
                continue

            _, mel_post, _ = infer_vid(images, model, mode='test')
            s = mel_post.squeeze(0).contiguous().cpu().detach().numpy()

            if w_idx == 0:
                mel = s
            else:
                mel = np.concatenate((mel, s[:, hps.mel_overlap:]), axis=1)

    if mel is None:
        print(f'  Skipped {interval_name} (inference failed)')
        continue

    # mel -> wav via Griffin-Lim
    wav = audio.inv_mel_spectrogram(mel, hps)
    audio.save_wav(wav, wav_out, sr=hps.sample_rate)

    # Copy ground-truth audio
    gt_audio = os.path.join(interval_dir, 'audio', 'audio.wav')
    if os.path.exists(gt_audio):
        shutil.copy2(gt_audio, gt_out)

    duration = len(wav) / hps.sample_rate
    print(f'  {interval_name}: mel {mel.shape}, '
          f'{duration:.1f}s, range [{mel.min():.2f}, {mel.max():.2f}]')

gen_wavs = sorted(glob(os.path.join(RESULTS_DIR, 'wavs', '*.wav')))
gt_wavs  = sorted(glob(os.path.join(RESULTS_DIR, 'gts', '*.wav')))
print(f'\nGenerated: {len(gen_wavs)} wavs | Ground truth: {len(gt_wavs)} wavs')

hparams: fps=30, T=90, overlap=15, img_size=96, mel_overlap=40
Loading model...
Model loaded (iteration 223000)

Processing 0 intervals...



Inference: 0it [00:00, ?it/s]


Generated: 0 wavs | Ground truth: 0 wavs


## Cell 8 — Listen: Generated vs Ground Truth

In [29]:
from IPython.display import Audio, display
from glob import glob
import os

gen_wavs = sorted(glob(os.path.join(RESULTS_DIR, 'wavs', '*.wav')))

for gen_path in gen_wavs[:3]:
    name = os.path.basename(gen_path)
    gt_path = os.path.join(RESULTS_DIR, 'gts', name)
    
    print(f'\n--- {name} ---')
    print('Generated:')
    display(Audio(gen_path))
    if os.path.exists(gt_path):
        print('Ground Truth:')
        display(Audio(gt_path))

## Cell 9 — Evaluate: PESQ, STOI, ESTOI

Compute the same metrics reported in the paper to verify reproduction.

In [30]:
import numpy as np
import librosa
from glob import glob
import os

try:
    from pesq import pesq
    from pystoi import stoi
    HAS_METRICS = True
except ImportError:
    print('Install: pip install pesq pystoi')
    HAS_METRICS = False

if HAS_METRICS:
    SR = 16000
    gen_wavs = sorted(glob(os.path.join(RESULTS_DIR, 'wavs', '*.wav')))
    pesq_scores, stoi_scores, estoi_scores = [], [], []
    
    for gen_path in gen_wavs:
        name = os.path.basename(gen_path)
        gt_path = os.path.join(RESULTS_DIR, 'gts', name)
        if not os.path.exists(gt_path):
            continue
        
        gen_wav, _ = librosa.load(gen_path, sr=SR)
        gt_wav, _  = librosa.load(gt_path, sr=SR)
        
        min_len = min(len(gen_wav), len(gt_wav))
        gen_wav, gt_wav = gen_wav[:min_len], gt_wav[:min_len]
        if min_len < SR:
            continue
        
        try:
            pesq_scores.append(pesq(SR, gt_wav, gen_wav, 'wb'))
        except Exception as e:
            print(f'  PESQ error on {name}: {e}')
        try:
            stoi_scores.append(stoi(gt_wav, gen_wav, SR, extended=False))
        except Exception as e:
            print(f'  STOI error on {name}: {e}')
        try:
            estoi_scores.append(stoi(gt_wav, gen_wav, SR, extended=True))
        except Exception as e:
            print(f'  ESTOI error on {name}: {e}')
    
    print('\n=== Your Reproduction ===')
    if pesq_scores:
        print(f'PESQ:  {np.mean(pesq_scores):.3f} +/- {np.std(pesq_scores):.3f}  (n={len(pesq_scores)})')
    if stoi_scores:
        print(f'STOI:  {np.mean(stoi_scores):.3f} +/- {np.std(stoi_scores):.3f}  (n={len(stoi_scores)})')
    if estoi_scores:
        print(f'ESTOI: {np.mean(estoi_scores):.3f} +/- {np.std(estoi_scores):.3f}  (n={len(estoi_scores)})')
    
    print('\n=== Paper Reported (chem speaker) ===')
    print('PESQ:  ~1.2-1.4')
    print('STOI:  ~0.45-0.55')
    print('ESTOI: ~0.28-0.35')


=== Your Reproduction ===

=== Paper Reported (chem speaker) ===
PESQ:  ~1.2-1.4
STOI:  ~0.45-0.55
ESTOI: ~0.28-0.35


## Cell 10 — Download Results

In [31]:
import shutil
shutil.make_archive('/content/lip2wav_results', 'zip', RESULTS_DIR)
print(f'Results: /content/lip2wav_results.zip')

from google.colab import files
files.download('/content/lip2wav_results.zip')

Results: /content/lip2wav_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
# Download all generated wavs, ground-truth wavs, and evaluation results to local machine
import shutil, os
from google.colab import files

# Bundle everything into a zip
RESULTS_DIR = '/content/results'
zip_path = '/content/lip2wav_results'

if os.path.exists(RESULTS_DIR) and (os.listdir(os.path.join(RESULTS_DIR, 'wavs', '')) if os.path.isdir(os.path.join(RESULTS_DIR, 'wavs')) else []):
    shutil.make_archive(zip_path, 'zip', RESULTS_DIR)
    print(f'Downloading {zip_path}.zip ({os.path.getsize(zip_path + ".zip") / 1024 / 1024:.1f} MB)')
    print(f'  Contains: wavs/ (generated), gts/ (ground truth)')
    files.download(zip_path + '.zip')
else:
    print('No results to download. Make sure inference (Cell 7b) ran successfully.')
    print('Check: valid_intervals was empty — preprocessing likely failed.')

No results to download. Make sure inference (Cell 7b) ran successfully.
Check: valid_intervals was empty — preprocessing likely failed.
